# 🏠 Định giá nhà đất Hà Nội bằng AI (LightGBM + PhoBERT)

Notebook này làm **toàn bộ dự án từ đầu đến cuối**: tải dữ liệu → làm sạch → fine-tune PhoBERT đọc mô tả tin rao →
huấn luyện LightGBM → đánh giá → **đưa giao diện web lên Hugging Face Spaces** (có link cố định để dán vào CV).

**Cách chạy:**
1. Menu **Runtime → Change runtime type → T4 GPU** → Save.
2. Menu **Runtime → Run all**.
3. Khi được hỏi, dán **Hugging Face token** (quyền *Write*) vào ô nhập rồi Enter.
4. Chờ khoảng 60–90 phút. Cuối notebook sẽ in ra link web demo và tải về gói file cho GitHub.

Dữ liệu: [tinixai/vietnam-real-estates](https://huggingface.co/datasets/tinixai/vietnam-real-estates) (CC BY-NC 4.0) —
tin rao bán bất động sản 06/2025 – 03/2026.

## 0. Cài thư viện & cấu hình

In [ ]:
!pip -q install lightgbm pyarrow "huggingface_hub>=0.24"

In [ ]:
import os, gc, json, math, time, random, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ===== CẤU HÌNH (có thể chỉnh) =====
SPACE_NAME      = "dinh-gia-nha-ha-noi"      # tên web demo trên Hugging Face
MODEL_NAME      = "vinai/phobert-base-v2"    # mô hình ngôn ngữ tiếng Việt
USE_PHOBERT     = True     # False = chỉ dùng LightGBM (chạy rất nhanh, dùng khi không có GPU)
N_PHOBERT_TRAIN = 80_000   # số tin dùng để fine-tune PhoBERT
MAX_LEN         = 160      # số token tối đa của mô tả
EPOCHS          = 2
BATCH_SIZE      = 32
LR              = 3e-5
TEST_FROM       = "2026-02-01"  # tin đăng từ ngày này trở đi dùng làm tập kiểm tra (mô hình chưa từng thấy)
SEED            = 42
LOCAL_TEST      = os.environ.get("LOCAL_TEST") == "1"   # chỉ dùng khi chạy thử ngoài Colab

random.seed(SEED); np.random.seed(SEED)
os.makedirs("space", exist_ok=True); os.makedirs("outputs", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "KHÔNG CÓ")
if USE_PHOBERT and not HAS_GPU and not LOCAL_TEST:
    print("⚠️ Không có GPU nên tự tắt PhoBERT. Muốn dùng PhoBERT: Runtime → Change runtime type → T4 GPU, rồi chạy lại.")
    USE_PHOBERT = False

## 1. File xử lý đặc trưng dùng chung (train + app)

In [ ]:
%%writefile space/features.py
"""
features.py — Xử lý dữ liệu & tạo đặc trưng, dùng CHUNG cho lúc train (Colab) và lúc chạy app (HF Space).
Giữ một file duy nhất để đảm bảo đặc trưng lúc train và lúc dự đoán giống hệt nhau.
"""
import re
import unicodedata

import numpy as np
import pandas as pd

RAW_COLS = [
    "name", "description", "property_type_name", "province_name", "district_name",
    "ward_name", "street_name", "project_name", "price", "area", "floor_count",
    "frontage_width", "house_depth", "road_width", "bedroom_count", "bathroom_count",
    "house_direction", "published_at",
]

LOAI_NHA = "Nhà riêng"
LOAI_CC = "Chung cư"
START_MONTH = pd.Timestamp("2025-06-01")


# ----------------------------------------------------------------------------- văn bản
def norm_text(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = unicodedata.normalize("NFC", str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()


# Che giá trong văn bản để mô hình KHÔNG "đọc trộm" đáp án (tránh rò rỉ dữ liệu)
_NUM = r"\d+(?:[.,]\d+)*"
_NL = r"(?![^\W\d_])"  # không được nối tiếp bằng chữ cái (tránh "5 tiện ích", "1 trệt")
_PRICE_PATTERNS = [
    rf"{_NUM}\s*(?:tỷ|tỉ|ty|ti){_NL}(?:\s*{_NUM})?",                            # 8 tỷ, 8,2 tỷ, 8 tỷ 2, 8ty2
    rf"{_NUM}\s*(?:triệu|trieu|tr|củ|cu){_NL}(?:\s*/\s*(?:m2|m²|m))?",          # 950 triệu, 120tr/m2
    rf"{_NUM}\s*(?:nghìn|ngàn|k)\s*/\s*(?:m2|m²|tháng)",                         # 500k/m2
    rf"{_NUM}\s*(?:vnđ|vnd|đồng|đ){_NL}",                                       # 7.400.000.000 đ
    r"giá\s*(?:chỉ|chào|bán|tốt|rẻ|:)*\s*\[giá\]",
]
_PRICE_RE = [re.compile(p) for p in _PRICE_PATTERNS]


def mask_price(text: str) -> str:
    t = norm_text(text)
    for rx in _PRICE_RE:
        t = rx.sub("[giá]", t)
    return t


def make_text(name, description, max_chars: int = 700) -> str:
    """Văn bản đưa vào PhoBERT: tiêu đề + mô tả, đã che giá."""
    t = f"{norm_text(name)} . {norm_text(description)}"
    return mask_price(t)[:max_chars]


# ----------------------------------------------------------------------------- từ khoá
# (tên cột, nhãn tiếng Việt, regex, cụm từ chèn vào khi người dùng tick ô trên giao diện)
KEYWORDS = [
    ("kw_mat_pho", "Mặt phố / mặt đường", r"mặt phố|mặt đường|mặt tiền phố", "mặt phố"),
    ("kw_oto", "Ô tô đỗ cửa / ngõ ô tô", r"ô ?tô|oto|xe hơi", "ô tô đỗ cửa"),
    ("kw_kinh_doanh", "Kinh doanh được", r"kinh doanh|buôn bán", "kinh doanh tốt"),
    ("kw_lo_goc", "Lô góc / căn góc", r"lô góc|căn góc|góc 2 mặt|hai mặt thoáng|2 mặt thoáng|3 mặt thoáng", "lô góc"),
    ("kw_thang_may", "Có thang máy", r"thang máy", "có thang máy"),
    ("kw_noi_that", "Full nội thất", r"full nội thất|đầy đủ nội thất|nội thất đầy đủ|nội thất cao cấp|nội thất xịn", "full nội thất"),
    ("kw_view", "View hồ / sông / công viên", r"view hồ|view sông|view công viên|gần hồ|hướng hồ", "view hồ"),
    ("kw_phan_lo", "Phân lô", r"phân lô", "phân lô"),
    ("kw_ngo_nong", "Ngõ nông, gần phố", r"ngõ nông|gần phố|cách phố|ra phố", "ngõ nông gần phố"),
    ("kw_ban_gap", "Cần bán gấp / cắt lỗ", r"bán gấp|cần bán gấp|cắt lỗ|ngộp|giảm sâu|hạ giá", "cần bán gấp"),
    ("kw_dong_tien", "Có dòng tiền cho thuê", r"dòng tiền|đang cho thuê|cho thuê \d", "đang cho thuê có dòng tiền"),
    ("kw_xay_moi", "Nhà mới xây", r"mới xây|xây mới|nhà mới|mới tinh", "nhà mới xây"),
]
KW_COLS = [k[0] for k in KEYWORDS]
KW_LABEL = {k[0]: k[1] for k in KEYWORDS}
KW_PHRASE = {k[1]: k[3] for k in KEYWORDS}
_KW_RE = {k[0]: re.compile(k[2]) for k in KEYWORDS}

PHAP_LY = ["Sổ đỏ/sổ hồng", "HĐMB/đang chờ sổ", "Giấy tờ khác", "Không rõ"]
PHAP_LY_PHRASE = {
    "Sổ đỏ/sổ hồng": "sổ đỏ chính chủ",
    "HĐMB/đang chờ sổ": "hợp đồng mua bán đang chờ sổ",
    "Giấy tờ khác": "giấy tờ viết tay",
    "Không rõ": "",
}
_RE_HDMB = re.compile(r"chưa có sổ|chờ sổ|đang làm sổ|hđmb|hợp đồng mua bán")
_RE_KHAC = re.compile(r"viết tay|sổ chung|vi bằng|giấy tờ tay|sổ chung")
_RE_SO = re.compile(r"sổ đỏ|sổ hồng|sđcc|sổ riêng|có sổ|sổ vuông|sổ đẹp|sổ nở hậu|sẵn sổ")


def phap_ly_of(text: str) -> str:
    if _RE_HDMB.search(text):
        return "HĐMB/đang chờ sổ"
    if _RE_KHAC.search(text):
        return "Giấy tờ khác"
    if _RE_SO.search(text):
        return "Sổ đỏ/sổ hồng"
    return "Không rõ"


# ----------------------------------------------------------------------------- chuẩn hoá cột
def map_loai(t) -> str | None:
    t = norm_text(t)
    if "chung cư" in t or "căn hộ" in t:
        return LOAI_CC
    if t in {"nhà", "nhà riêng", "nhà mặt phố", "nhà trong ngõ", "nhà phố", "nhà mặt tiền", "nhà hẻm"}:
        return LOAI_NHA
    return None  # đất, biệt thự/liền kề, shophouse... -> bỏ


HUONG = ["Đông", "Tây", "Nam", "Bắc", "Đông Bắc", "Đông Nam", "Tây Bắc", "Tây Nam"]


def norm_huong(h) -> str:
    t = norm_text(h).replace("-", " ").replace("_", " ")
    t = re.sub(r"\s+", " ", t).strip()
    for v in HUONG:
        if t == v.lower():
            return v
    return "Không rõ"


def clean_place(s) -> str:
    t = unicodedata.normalize("NFC", str(s)).strip() if s is not None and not (isinstance(s, float) and np.isnan(s)) else ""
    t = re.sub(r"^(quận|huyện|thị xã|phường|xã|thị trấn)\s+", "", t, flags=re.I).strip()
    return t if t else "Không rõ"


# ----------------------------------------------------------------------------- làm sạch (chỉ dùng lúc train)
def clean_listings(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    log = []
    n0 = len(df)
    df = df.copy()
    df["loai"] = df["property_type_name"].map(map_loai)
    df = df[df["loai"].notna()]
    log.append(("Giữ nhà riêng + chung cư", len(df)))

    for c in ["price", "area", "floor_count", "frontage_width", "house_depth", "road_width", "bedroom_count", "bathroom_count"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df[df["price"].notna() & df["area"].notna() & (df["price"] > 0)]
    log.append(("Có giá & diện tích", len(df)))

    a_ok = np.where(df["loai"] == LOAI_CC, df["area"].between(20, 300), df["area"].between(12, 500))
    df = df[a_ok]
    df["ppm2"] = df["price"] / df["area"]
    df = df[df["ppm2"].between(15e6, 1.5e9)]  # 15 triệu – 1,5 tỷ / m²
    log.append(("Diện tích & đơn giá hợp lý", len(df)))

    # loại trùng lặp (tin đăng lại nhiều lần) — giữ tin đăng sớm nhất
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at")
    df["_k1"] = df["loai"] + "|" + df["district_name"].astype(str) + "|" + df["area"].round(0).astype(str) + "|" + df["price"].round(-7).astype(str)
    df["_k2"] = df["description"].map(lambda s: norm_text(s)[:300])
    df = df.drop_duplicates("_k1")
    df = df[(df["_k2"].str.len() < 30) | ~df["_k2"].duplicated()]
    log.append(("Bỏ tin trùng lặp", len(df)))

    # bỏ ngoại lai theo từng (loại, quận): ngoài khoảng 1%–99% log đơn giá
    df["y"] = np.log(df["ppm2"])
    g = df.groupby(["loai", "district_name"], dropna=False)["y"]
    lo, hi = g.transform(lambda s: s.quantile(0.01)), g.transform(lambda s: s.quantile(0.99))
    df = df[(df["y"] >= lo) & (df["y"] <= hi)]
    log.append(("Bỏ ngoại lai theo quận", len(df)))

    df = df.drop(columns=["_k1", "_k2"]).reset_index(drop=True)
    if verbose:
        print(f"Ban đầu: {n0:,}")
        for k, v in log:
            print(f"  → {k}: {v:,}")
    return df


# ----------------------------------------------------------------------------- đặc trưng
NUM_COLS = [
    "area", "log_area", "floor_count", "san_xd", "frontage_width", "house_depth", "road_width",
    "bedroom_count", "bathroom_count", "month_idx",
]
CAT_COLS = ["loai", "district", "ward", "project", "huong", "phap_ly"]


def base_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo các cột đặc trưng 'thô' (chưa gắn danh mục)."""
    out = pd.DataFrame(index=df.index)
    loai = df["loai"] if "loai" in df else df["property_type_name"].map(map_loai)
    out["loai"] = loai.fillna(LOAI_NHA)
    is_cc = out["loai"] == LOAI_CC
    num = lambda c: pd.to_numeric(df.get(c), errors="coerce") if c in df else pd.Series(np.nan, index=df.index)

    out["area"] = num("area")
    out["log_area"] = np.log(out["area"].clip(lower=1))
    fl = num("floor_count").where(lambda s: s.between(1, 15))
    out["floor_count"] = fl.where(~is_cc)
    out["san_xd"] = (out["area"] * out["floor_count"]).where(~is_cc)
    out["frontage_width"] = num("frontage_width").where(lambda s: s.between(1, 30)).where(~is_cc)
    out["house_depth"] = num("house_depth").where(lambda s: s.between(2, 100)).where(~is_cc)
    out["road_width"] = num("road_width").where(lambda s: s.between(0.5, 60)).where(~is_cc)
    out["bedroom_count"] = num("bedroom_count").where(lambda s: s.between(0, 15))
    out["bathroom_count"] = num("bathroom_count").where(lambda s: s.between(0, 15))

    pub = pd.to_datetime(df["published_at"], errors="coerce") if "published_at" in df else pd.Series(pd.NaT, index=df.index)
    out["month_idx"] = ((pub.dt.year - START_MONTH.year) * 12 + (pub.dt.month - START_MONTH.month)).astype(float)

    text = (df.get("name", "").fillna("").astype(str) + " . " + df.get("description", "").fillna("").astype(str)).map(norm_text)
    for col, rx in _KW_RE.items():
        out[col] = text.str.contains(rx).astype(np.int8)
    out["phap_ly"] = text.map(phap_ly_of)

    out["district"] = df["district_name"].map(clean_place)
    out["ward"] = out["district"] + " | " + df["ward_name"].map(clean_place)
    proj = df["project_name"].map(lambda s: unicodedata.normalize("NFC", str(s)).strip() if isinstance(s, str) and s.strip() else "Không rõ")
    out["project"] = proj.where(is_cc, "Không áp dụng")
    out["huong"] = df["house_direction"].map(norm_huong) if "house_direction" in df else "Không rõ"
    return out


def fit_categories(F: pd.DataFrame, min_count: dict | None = None) -> dict:
    """Lưu danh mục hợp lệ (đủ số mẫu); nhóm hiếm gộp thành 'Khác'."""
    min_count = min_count or {"district": 30, "ward": 30, "project": 30}
    cats = {}
    for c in CAT_COLS:
        vc = F[c].value_counts()
        keep = vc[vc >= min_count.get(c, 1)].index.tolist()
        cats[c] = sorted(set(keep) | {"Khác"})
    return cats


def apply_categories(F: pd.DataFrame, cats: dict) -> pd.DataFrame:
    F = F.copy()
    for c in CAT_COLS:
        v = F[c].where(F[c].isin(cats[c]), "Khác")
        F[c] = pd.Categorical(v, categories=cats[c])
    return F


def feature_list(use_text_kw: bool = True, use_phobert: bool = False) -> list:
    cols = NUM_COLS + CAT_COLS
    if use_text_kw:
        cols = cols + KW_COLS
    else:
        cols = [c for c in cols if c != "phap_ly"]
    if use_phobert:
        cols = cols + ["phobert"]
    return cols


# Nhóm đặc trưng để giải thích kết quả trên giao diện
def group_of(col: str) -> str:
    if col in ("district", "ward"):
        return "Vị trí (quận, phường)"
    if col == "project":
        return "Dự án chung cư"
    if col in ("area", "log_area", "san_xd"):
        return "Diện tích"
    if col == "floor_count":
        return "Số tầng"
    if col in ("frontage_width", "house_depth"):
        return "Mặt tiền & chiều sâu"
    if col == "road_width":
        return "Độ rộng ngõ/đường"
    if col in ("bedroom_count", "bathroom_count"):
        return "Số phòng"
    if col == "huong":
        return "Hướng nhà"
    if col == "phap_ly":
        return "Pháp lý"
    if col == "loai":
        return "Loại bất động sản"
    if col == "month_idx":
        return "Thời điểm"
    if col == "phobert":
        return "Nội dung mô tả (PhoBERT)"
    if col in KW_LABEL:
        return KW_LABEL[col]
    return col


In [ ]:
import sys, importlib
sys.path.insert(0, "space")
import features; importlib.reload(features)
from features import *

## 2. Đăng nhập Hugging Face
Lấy token tại https://huggingface.co/settings/tokens → **Create new token** → chọn loại **Write** → copy.
(Hoặc lưu sẵn trong Colab: biểu tượng 🔑 Secrets bên trái, tên `HF_TOKEN`.)

In [ ]:
from huggingface_hub import login, whoami
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and not LOCAL_TEST:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Dán Hugging Face token (quyền Write) rồi Enter: ").strip()
if not LOCAL_TEST:
    login(token=HF_TOKEN, add_to_git_credential=False)
    HF_USER = whoami(token=HF_TOKEN)["name"]
    print("✅ Đã đăng nhập:", HF_USER)
else:
    HF_USER = "local"

In [ ]:
# Kiểm tra nhanh tokenizer PhoBERT (để nếu lỗi thì biết ngay từ đầu)
if USE_PHOBERT and not LOCAL_TEST:
    from transformers import AutoTokenizer
    _t = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(_t.tokenize("bán nhà mặt phố đống đa , ô tô đỗ cửa , sổ đỏ chính chủ"))

## 3. Tải dữ liệu & lọc Hà Nội

In [ ]:
import pyarrow.dataset as pads
import pyarrow.compute as pc

if LOCAL_TEST:
    raw = pd.read_parquet(os.environ["LOCAL_DATA"])
else:
    from huggingface_hub import snapshot_download
    path = snapshot_download("tinixai/vietnam-real-estates", repo_type="dataset",
                             allow_patterns=["*.parquet"], local_dir="data_raw", token=HF_TOKEN)
    dset = pads.dataset(path, format="parquet")
    cols = [c for c in RAW_COLS if c in dset.schema.names]
    prov = pc.unique(dset.to_table(columns=["province_name"])["province_name"]).to_pylist()
    hn = [p for p in prov if isinstance(p, str) and "nội" in p.lower() and "hà" in p.lower()]
    print("Tên tỉnh khớp Hà Nội:", hn)
    raw = dset.to_table(columns=cols, filter=pads.field("province_name").isin(hn)).to_pandas()
print(f"Số tin rao tại Hà Nội: {len(raw):,}")
print(raw["property_type_name"].value_counts())

## 4. Làm sạch dữ liệu

In [ ]:
df = clean_listings(raw)
del raw; gc.collect()
print(df["loai"].value_counts())
print("Khoảng thời gian:", df["published_at"].min().date(), "→", df["published_at"].max().date())

In [ ]:
# Biểu đồ: đơn giá trung vị theo quận (nhà riêng)
nha = df[df.loai == LOAI_NHA]
top = nha.groupby("district_name")["ppm2"].agg(["median", "size"]).query("size >= 200").sort_values("median").tail(20)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top["median"] / 1e6, color="#0f766e")
ax.set_xlabel("Đơn giá trung vị (triệu đồng/m²)")
ax.set_title("Nhà riêng Hà Nội — đơn giá rao bán trung vị theo quận")
plt.tight_layout(); plt.savefig("outputs/don_gia_theo_quan.png"); plt.show()

## 5. Chia tập train / val / test theo thời gian
Tập **test là các tin đăng sau ngày `TEST_FROM`** — giống thực tế: học từ dữ liệu quá khứ, định giá tin mới.
Tập train được chia đôi: phần A để fine-tune PhoBERT, phần B để train LightGBM (tránh rò rỉ khi xếp chồng mô hình).

In [ ]:
rng = np.random.default_rng(SEED)
df["split"] = np.where(df["published_at"] >= pd.Timestamp(TEST_FROM), "test", "train")
if (df.split == "test").mean() < 0.03:  # phòng khi dữ liệu không có đủ tin sau TEST_FROM
    df["split"] = np.where(rng.random(len(df)) < 0.12, "test", "train")
tr = df.index[df.split == "train"]
df.loc[tr[rng.random(len(tr)) < 0.08], "split"] = "val"
tr = df.index[df.split == "train"]
nA = min(N_PHOBERT_TRAIN, int(0.35 * len(tr))) if USE_PHOBERT else 0
A = rng.choice(tr, nA, replace=False) if nA else np.array([], dtype=int)
df.loc[tr, "split"] = "trainB"
df.loc[A, "split"] = "trainA"
print(df["split"].value_counts())

idxA, idxB = df.index[df.split == "trainA"], df.index[df.split == "trainB"]
idxV, idxT = df.index[df.split == "val"], df.index[df.split == "test"]
idxTrain = idxA.union(idxB)
y = df["y"]

## 6. Tạo đặc trưng
Thông số (diện tích, số tầng, mặt tiền, ngõ...), vị trí (quận, phường, dự án), và **từ khoá trích từ mô tả**
(ô tô đỗ cửa, kinh doanh, lô góc, pháp lý...).

In [ ]:
F_raw = base_frame(df)
CATS = fit_categories(F_raw.loc[idxTrain])
F = apply_categories(F_raw, CATS)
print("Số đặc trưng:", F.shape[1])
print("Tỷ lệ tin có từ khoá:")
print((F[KW_COLS].mean() * 100).round(1).rename(KW_LABEL).sort_values(ascending=False).to_string())
print(F["phap_ly"].value_counts(normalize=True).round(3))

## 7. Chuẩn bị văn bản — **che giá trong mô tả** để tránh rò rỉ dữ liệu
Rất nhiều tin ghi sẵn giá ("giá 7,4 tỷ") trong tiêu đề/mô tả. Nếu không che, mô hình chỉ cần "đọc" giá là ra đáp án → kết quả đẹp giả.

In [ ]:
df["text"] = [make_text(n, d) for n, d in zip(df["name"], df["description"])]
for i in df.sample(3, random_state=1).index:
    print("GỐC :", str(df.at[i, "name"])[:150])
    print("CHE :", df.at[i, "text"][:150], "\n")
import re
leak = df["text"].str.contains(r"\d\s*(?:tỷ|tỉ)(?![^\W\d_])").mean()
print(f"Tỷ lệ văn bản còn sót con số 'tỷ' sau khi che: {leak:.2%}")

## 8. Fine-tune PhoBERT đọc mô tả → dự đoán đơn giá
(Bỏ qua nếu `USE_PHOBERT = False`. Trên GPU T4 mất khoảng 30–45 phút.)

In [ ]:
if USE_PHOBERT:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    if LOCAL_TEST:
        tok, model = LOCAL_TOKENIZER, LOCAL_MODEL
    else:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    model.to(DEVICE)
    Y_MU, Y_SD = float(y[idxA].mean()), float(y[idxA].std())

    def encode(texts):
        out = []
        for i in range(0, len(texts), 2000):
            out += tok(list(texts[i:i + 2000]), truncation=True, max_length=MAX_LEN)["input_ids"]
        return out

    def collate(batch_ids):
        pad = tok.pad_token_id
        L = max(len(x) for x in batch_ids)
        ids = torch.full((len(batch_ids), L), pad, dtype=torch.long)
        for j, x in enumerate(batch_ids):
            ids[j, :len(x)] = torch.tensor(x)
        return ids.to(DEVICE), (ids != pad).long().to(DEVICE)

    @torch.no_grad()
    def predict_text(idx, bs=128, chunk=20_000):
        model.eval()
        res = []
        for c0 in range(0, len(idx), chunk):            # xử lý từng khối để không tràn RAM
            sub = idx[c0:c0 + chunk]
            ids = encode(df.loc[sub, "text"].tolist())
            order = np.argsort([len(x) for x in ids])
            out = np.zeros(len(ids), dtype=np.float32)
            for s in range(0, len(ids), bs):
                b = order[s:s + bs]
                inp, att = collate([ids[k] for k in b])
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=HAS_GPU):
                    out[b] = model(input_ids=inp, attention_mask=att).logits.float().reshape(-1).cpu().numpy()
            res.append(pd.Series(out, index=sub))
            if len(idx) > chunk:
                print(f"    dự đoán {min(c0 + chunk, len(idx)):,}/{len(idx):,}")
        return pd.concat(res)

    t0 = time.time()
    idsA = encode(df.loc[idxA, "text"].tolist())
    tgtA = ((y[idxA].values - Y_MU) / Y_SD).astype(np.float32)
    print(f"Tokenize xong {len(idsA):,} văn bản trong {time.time() - t0:.0f}s")

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(idsA) / BATCH_SIZE)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=HAS_GPU)
    loss_fn = torch.nn.SmoothL1Loss(beta=1.0)
    idxV_small = idxV[:5000]
    step = 0
    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(len(idsA))
        run = 0.0
        for s in range(0, len(perm), BATCH_SIZE):
            b = perm[s:s + BATCH_SIZE]
            inp, att = collate([idsA[k] for k in b])
            tgt = torch.tensor(tgtA[b], device=DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=HAS_GPU):
                pred = model(input_ids=inp, attention_mask=att).logits.float().reshape(-1)
                loss = loss_fn(pred, tgt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run = 0.98 * run + 0.02 * loss.item() if step else loss.item()
            step += 1
            if step % 300 == 0:
                print(f"  epoch {ep + 1} | bước {step}/{steps} | loss {run:.4f} | {time.time() - t0:.0f}s")
        pv = predict_text(idxV_small) * Y_SD + Y_MU
        mdape = np.median(np.abs(np.exp(pv - y[idxV_small]) - 1)) * 100
        print(f"✅ Hết epoch {ep + 1}: sai số trung vị (val) của riêng PhoBERT = {mdape:.1f}%")

    # Dự đoán cho tập B / val / test → làm đặc trưng cho LightGBM
    idx_need = idxB.union(idxV).union(idxT)
    F["phobert"] = np.nan
    F.loc[idx_need, "phobert"] = predict_text(idx_need)
    print(f"PhoBERT xong sau {(time.time() - t0) / 60:.1f} phút")

    # Lưu mô hình (fp16 cho nhẹ)
    model.to("cpu").half().save_pretrained("space/phobert")
    tok.save_pretrained("space/phobert")
    del model, opt; gc.collect()
    if HAS_GPU: torch.cuda.empty_cache()

## 9. Huấn luyện LightGBM & so sánh các mô hình

In [ ]:
import lightgbm as lgb

def train_lgb(cols, idx_tr, idx_va, rounds=4000):
    cats = [c for c in cols if c in CAT_COLS]
    dtr = lgb.Dataset(F.loc[idx_tr, cols], y[idx_tr], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(F.loc[idx_va, cols], y[idx_va], categorical_feature=cats, reference=dtr)
    params = dict(objective="regression", metric="l1", learning_rate=0.05, num_leaves=255,
                  min_data_in_leaf=40, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l2=1.0, cat_smooth=20, cat_l2=10, max_cat_to_onehot=4, verbose=-1, seed=SEED)
    return lgb.train(params, dtr, rounds, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(500)])

def metrics(name, y_true, y_pred):
    ape = np.abs(np.exp(y_pred - y_true) - 1) * 100       # sai số % của giá (= sai số % của đơn giá)
    return dict(model=name, MdAPE=float(np.median(ape)), MAPE=float(np.mean(ape)),
                within10=float((ape <= 10).mean() * 100), within20=float((ape <= 20).mean() * 100),
                R2_log=float(1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)))

results, models = [], {}
yT = y[idxT].values

t0 = time.time()
cols1 = feature_list(use_text_kw=False)
models["M1"] = train_lgb(cols1, idxTrain, idxV)
results.append(metrics("LightGBM – chỉ thông số", yT, models["M1"].predict(F.loc[idxT, cols1])))

cols2 = feature_list(use_text_kw=True)
models["M2"] = train_lgb(cols2, idxTrain, idxV)
results.append(metrics("LightGBM + từ khoá mô tả", yT, models["M2"].predict(F.loc[idxT, cols2])))

if USE_PHOBERT:
    results.append(metrics("PhoBERT – chỉ đọc mô tả", yT, F.loc[idxT, "phobert"].values * Y_SD + Y_MU))
    cols4 = feature_list(use_text_kw=True, use_phobert=True)
    models["M4"] = train_lgb(cols4, idxB, idxV)
    results.append(metrics("LightGBM + từ khoá + PhoBERT", yT, models["M4"].predict(F.loc[idxT, cols4])))
    # chọn mô hình cuối dựa trên tập VALIDATION (không nhìn tập test)
    v2 = metrics("M2", y[idxV].values, models["M2"].predict(F.loc[idxV, cols2]))["MdAPE"]
    v4 = metrics("M4", y[idxV].values, models["M4"].predict(F.loc[idxV, cols4]))["MdAPE"]
    print(f"Sai số trung vị trên val: không PhoBERT {v2:.2f}% | có PhoBERT {v4:.2f}%")
    if v4 > v2:
        print("→ PhoBERT không giúp cải thiện trên val, mô hình cuối dùng LightGBM + từ khoá.")
        USE_PHOBERT = False
        shutil.rmtree("space/phobert", ignore_errors=True)
if USE_PHOBERT:
    FINAL, FINAL_COLS, FINAL_NAME = models["M4"], cols4, "LightGBM + từ khoá + PhoBERT"
else:
    FINAL, FINAL_COLS, FINAL_NAME = models["M2"], cols2, "LightGBM + từ khoá mô tả"

res = pd.DataFrame(results).set_index("model").round(2)
print("Mô hình cuối:", FINAL_NAME)
print(f"Train LightGBM xong sau {(time.time() - t0) / 60:.1f} phút\n")
print("KẾT QUẢ TRÊN TẬP TEST (tin đăng sau", TEST_FROM, ")")
print(res.to_string())

In [ ]:
# Kết quả theo loại BĐS (mô hình cuối)
predT = pd.Series(FINAL.predict(F.loc[idxT, FINAL_COLS]), index=idxT)
by_type = pd.DataFrame([metrics(l, y[g].values, predT[g].values) for l, g in df.loc[idxT].groupby("loai").groups.items()]).set_index("model").round(2)
print(by_type.to_string())

## 10. Biểu đồ kết quả (dùng cho README/CV)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
colors = ["#0f766e" if m == FINAL_NAME else "#94a3b8" for m in res.index]
ax.barh(res.index, res["MdAPE"], color=colors)
for i, v in enumerate(res["MdAPE"]):
    ax.text(v + 0.2, i, f"{v:.1f}%", va="center")
ax.invert_yaxis(); ax.set_xlabel("Sai số trung vị trên tập test (%) — càng thấp càng tốt")
ax.set_title("So sánh các mô hình định giá")
plt.tight_layout(); plt.savefig("outputs/so_sanh_mo_hinh.png"); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
s = rng.choice(len(idxT), min(4000, len(idxT)), replace=False)
pt, pp = np.exp(yT[s]) / 1e6, np.exp(predT.values[s]) / 1e6
ax.scatter(pt, pp, s=4, alpha=0.3, color="#0f766e")
lim = [min(pt.min(), pp.min()), max(pt.max(), pp.max())]
ax.plot(lim, lim, "k--", lw=1); ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Đơn giá thực tế (triệu/m²)"); ax.set_ylabel("Đơn giá dự đoán (triệu/m²)")
ax.set_title("Dự đoán vs thực tế (tập test)")
plt.tight_layout(); plt.savefig("outputs/du_doan_vs_thuc_te.png"); plt.show()

imp = pd.Series(FINAL.feature_importance("gain"), index=FINAL_COLS)
imp = imp.groupby(imp.index.map(group_of)).sum().sort_values().tail(12)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(imp.index, imp / imp.sum() * 100, color="#0f766e")
ax.set_xlabel("Mức độ quan trọng (%)"); ax.set_title("Yếu tố ảnh hưởng đến giá nhiều nhất")
plt.tight_layout(); plt.savefig("outputs/muc_do_quan_trong.png"); plt.show()

## 11. Lưu mô hình & dữ liệu cho web demo

In [ ]:
FINAL.save_model("space/model_lgb.txt")

# khoảng giá 80%: từ phân phối sai số trên tập val
predV = pd.Series(FINAL.predict(F.loc[idxV, FINAL_COLS]), index=idxV)
resid = y[idxV] - predV
interval = {l: {"q10": float(resid[g].quantile(0.10)), "q90": float(resid[g].quantile(0.90))}
            for l, g in df.loc[idxV].groupby("loai").groups.items()}
for l in (LOAI_NHA, LOAI_CC):
    interval.setdefault(l, {"q10": float(resid.quantile(0.1)), "q90": float(resid.quantile(0.9))})
cov = {l: float(((y[g] - predT[g] >= interval[l]["q10"]) & (y[g] - predT[g] <= interval[l]["q90"])).mean())
       for l, g in df.loc[idxT].groupby("loai").groups.items()}
print("Độ phủ thực tế của khoảng giá 80% trên tập test:", {k: f"{v:.0%}" for k, v in cov.items()})

base = df.loc[idxTrain].assign(district=F_raw.loc[idxTrain, "district"], ward=F_raw.loc[idxTrain, "ward"])
med_d = base.groupby(["loai", "district"])["ppm2"].agg(["median", "size"]).query("size >= 30")["median"]
med_w = base.groupby(["loai", "ward"])["ppm2"].agg(["median", "size"]).query("size >= 20")["median"]
median_ppm2 = {"district": {f"{a}|{b}": float(v) for (a, b), v in med_d.items()},
               "ward": {f"{a}|{b}": float(v) for (a, b), v in med_w.items()},
               "all": {l: float(v) for l, v in base.groupby("loai")["ppm2"].median().items()}}

vc_d = F_raw.loc[idxTrain, "district"].value_counts()
districts = [d for d in vc_d.index if d in CATS["district"] and d not in ("Khác", "Không rõ")]
wards_by_district, projects_by_district = {}, {}
for w in CATS["ward"]:
    if " | " in w:
        d, ww = w.split(" | ", 1)
        if ww not in ("Không rõ", "Khác"):
            wards_by_district.setdefault(d, []).append(ww)
pj = F_raw.loc[idxTrain].query("project not in ['Không rõ', 'Không áp dụng']")
pj = pj[pj["project"].isin(CATS["project"])]
for d, g in pj.groupby("district"):
    projects_by_district[d] = g["project"].value_counts().index.tolist()[:80]

meta = dict(
    cats=CATS, features=FINAL_COLS, final_model=FINAL_NAME, use_phobert=bool(USE_PHOBERT), max_len=MAX_LEN,
    interval=interval, interval_coverage=cov, median_ppm2=median_ppm2,
    metrics=res.reset_index().to_dict("records"), metrics_by_type=by_type.reset_index().to_dict("records"),
    districts=districts, wards_by_district={k: sorted(v) for k, v in wards_by_district.items()},
    projects_by_district=projects_by_district,
    latest_date=str(df["published_at"].max().date()),
    data_info=dict(n_train=int(len(idxTrain) + len(idxV)), n_test=int(len(idxT)), test_from=TEST_FROM,
                   date_from=str(df["published_at"].min().date()), date_to=str(df["published_at"].max().date())),
)
if USE_PHOBERT:
    meta.update(y_mu=Y_MU, y_sd=Y_SD, phobert_model=MODEL_NAME)
with open("space/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False)

# tin rao để hiển thị "tin tương tự" (ưu tiên tin mới)
comps = pd.DataFrame({
    "loai": df["loai"], "district": F_raw["district"], "ward": df["ward_name"].map(clean_place),
    "project": F_raw["project"], "area": df["area"], "price": df["price"], "ppm2": df["ppm2"],
    "title": df["name"].fillna("").astype(str).str.slice(0, 110), "published": df["published_at"].dt.strftime("%d/%m/%Y"),
    "_t": df["published_at"],
}).sort_values("_t", ascending=False).head(120_000).drop(columns="_t")
comps.to_parquet("space/comps.parquet", index=False)
print("Đã lưu:", os.listdir("space"))

## 12. File giao diện web (Gradio)

In [ ]:
%%writefile space/app.py
"""
app.py — Giao diện định giá nhà đất Hà Nội (Gradio).
Mô hình: LightGBM (thông số + từ khoá) kết hợp PhoBERT đọc phần mô tả tin rao.
"""
import json
import math
import os
import unicodedata

import gradio as gr
import lightgbm as lgb
import numpy as np
import pandas as pd

from features import (
    CAT_COLS, HUONG, KEYWORDS, KW_PHRASE, LOAI_CC, LOAI_NHA, PHAP_LY, PHAP_LY_PHRASE,
    apply_categories, base_frame, group_of, make_text,
)

HERE = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(HERE, "meta.json"), encoding="utf-8") as f:
    META = json.load(f)
BOOSTER = lgb.Booster(model_file=os.path.join(HERE, "model_lgb.txt"))
COMPS = pd.read_parquet(os.path.join(HERE, "comps.parquet"))
CATS, FEATS = META["cats"], META["features"]

# ----------------------------------------------------------------------------- PhoBERT
PHOBERT = None
if META.get("use_phobert"):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    _p = os.path.join(HERE, "phobert")
    _tok = AutoTokenizer.from_pretrained(_p)
    _model = AutoModelForSequenceClassification.from_pretrained(_p).float().eval()
    PHOBERT = (_tok, _model)


def phobert_score(text: str) -> float:
    import torch

    tok, model = PHOBERT
    with torch.no_grad():
        enc = tok([text], truncation=True, max_length=META["max_len"], return_tensors="pt")
        return float(model(**enc).logits.reshape(-1)[0])


# ----------------------------------------------------------------------------- định dạng số
def vn(x: float, d: int = 1) -> str:
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return s[:-2] if s.endswith(",0") else s


def fmt_price(v: float) -> str:
    return f"{vn(v / 1e9, 2)} tỷ" if v >= 1e9 else f"{vn(v / 1e6, 0)} triệu"


def fmt_ppm2(v: float) -> str:
    return f"{vn(v / 1e6, 1)} triệu/m²"


# ----------------------------------------------------------------------------- dự đoán
def build_input(loai, quan, phuong, du_an, dien_tich, so_tang, mat_tien, duong, ngu, wc, huong, phap_ly, dac_diem, mo_ta):
    is_cc = loai == LOAI_CC
    phrases = [KW_PHRASE[d] for d in (dac_diem or [])]
    if PHAP_LY_PHRASE.get(phap_ly):
        phrases.append(PHAP_LY_PHRASE[phap_ly])
    # tiêu đề "giả lập" giống một tin rao, để PhoBERT có ngữ cảnh khi người dùng không viết mô tả
    if is_cc:
        title = f"Bán căn hộ chung cư {du_an + ' ' if du_an and du_an != 'Không rõ' else ''}{vn(dien_tich, 0)}m2 {int(ngu or 0)} ngủ {quan}"
    else:
        title = f"Bán nhà {quan} {vn(dien_tich, 0)}m2 {int(so_tang or 0)} tầng mặt tiền {vn(mat_tien or 0)}m"
    if phrases:
        title += ", " + ", ".join(phrases)
    none_if = lambda v, bad=("Không rõ", "", None): None if v in bad else v
    row = {
        "name": title,
        "description": (mo_ta or "").strip(),
        "property_type_name": "Căn hộ chung cư" if is_cc else "Nhà",
        "loai": loai,
        "district_name": quan,
        "ward_name": none_if(phuong),
        "project_name": none_if(du_an) if is_cc else None,
        "area": dien_tich,
        "floor_count": None if is_cc else so_tang,
        "frontage_width": None if is_cc else (mat_tien or None),
        "house_depth": None,
        "road_width": None if is_cc else (duong or None),
        "bedroom_count": ngu,
        "bathroom_count": wc,
        "house_direction": none_if(huong),
        "published_at": META["latest_date"],
    }
    return pd.DataFrame([row])


def predict(loai, quan, phuong, du_an, dien_tich, so_tang, mat_tien, duong, ngu, wc, huong, phap_ly, dac_diem, mo_ta):
    if not dien_tich or dien_tich <= 0:
        raise gr.Error("Vui lòng nhập diện tích lớn hơn 0.")
    df1 = build_input(loai, quan, phuong, du_an, dien_tich, so_tang, mat_tien, duong, ngu, wc, huong, phap_ly, dac_diem, mo_ta)
    F = apply_categories(base_frame(df1), CATS)
    if META.get("use_phobert"):
        F["phobert"] = phobert_score(make_text(df1.at[0, "name"], df1.at[0, "description"]))
    X = F[FEATS]
    y = float(BOOSTER.predict(X)[0])
    contrib = BOOSTER.predict(X, pred_contrib=True)[0]

    ppm2 = math.exp(y)
    price = ppm2 * dien_tich
    q = META["interval"][loai]
    lo, hi = math.exp(y + q["q10"]) * dien_tich, math.exp(y + q["q90"]) * dien_tich
    return card_html(loai, quan, phuong, price, lo, hi, ppm2), factors_html(contrib, loai), comps_table(loai, quan, phuong, du_an, dien_tich)


# ----------------------------------------------------------------------------- hiển thị
def card_html(loai, quan, phuong, price, lo, hi, ppm2):
    med = META["median_ppm2"]
    ref = []
    k_w = f"{loai}|{quan} | {phuong}"
    if phuong and phuong != "Không rõ" and k_w in med["ward"]:
        ref.append((f"Trung vị phường {phuong}", med["ward"][k_w]))
    if f"{loai}|{quan}" in med["district"]:
        ref.append((f"Trung vị {quan}", med["district"][f"{loai}|{quan}"]))
    ref.append((f"Trung vị {loai.lower()} toàn Hà Nội", med["all"][loai]))
    rows = ""
    for lbl, v in ref:
        d = ppm2 / v - 1
        color = "#16a34a" if d < 0 else "#dc2626"
        rows += (
            f"<tr><td style='padding:4px 12px 4px 0'>{lbl}</td><td style='padding:4px 12px'>{fmt_ppm2(v)}</td>"
            f"<td style='color:{color};font-weight:600'>{'+' if d >= 0 else ''}{vn(d * 100, 0)}%</td></tr>"
        )
    return f"""
<div style="border:1px solid var(--border-color-primary);border-radius:14px;padding:20px 22px">
  <div style="opacity:.7;font-size:14px">Giá ước tính</div>
  <div style="font-size:40px;font-weight:700;line-height:1.15;margin:4px 0">{fmt_price(price)}</div>
  <div style="font-size:15px">Khoảng giá hợp lý (80%): <b>{fmt_price(lo)} – {fmt_price(hi)}</b></div>
  <div style="font-size:15px;margin-top:2px">Đơn giá: <b>{fmt_ppm2(ppm2)}</b></div>
  <div style="margin-top:14px;font-size:14px;opacity:.85">So sánh đơn giá của bạn với thị trường:</div>
  <table style="font-size:14px;margin-top:4px">{rows}</table>
</div>"""


def factors_html(contrib, loai):
    groups = {}
    for f, c in zip(FEATS, contrib[:-1]):
        g = group_of(f)
        if f in ("loai", "project"):
            g = "Dự án chung cư" if (loai == LOAI_CC and f == "project") else "Loại bất động sản"
        groups[g] = groups.get(g, 0.0) + float(c)
    items = [(g, c) for g, c in groups.items() if abs(c) >= 0.01]
    items.sort(key=lambda t: -abs(t[1]))
    items = items[:9]
    base = math.exp(float(contrib[-1]))
    mx = max([abs(math.exp(c) - 1) for _, c in items] + [0.05])
    bars = ""
    for g, c in items:
        pct = math.exp(c) - 1
        w = min(100, abs(pct) / mx * 100)
        color = "#16a34a" if pct >= 0 else "#dc2626"
        bars += f"""
<div style="display:flex;align-items:center;gap:10px;margin:6px 0;font-size:14px">
  <div style="width:210px;flex-shrink:0">{g}</div>
  <div style="flex:1;background:var(--background-fill-secondary);border-radius:6px;height:18px;position:relative">
    <div style="width:{w:.0f}%;background:{color};height:18px;border-radius:6px;opacity:.85"></div>
  </div>
  <div style="width:70px;text-align:right;font-weight:600;color:{color}">{'+' if pct >= 0 else ''}{vn(pct * 100, 0)}%</div>
</div>"""
    return f"""
<div style="border:1px solid var(--border-color-primary);border-radius:14px;padding:18px 22px">
  <div style="font-weight:600;margin-bottom:4px">Yếu tố làm giá cao / thấp hơn</div>
  <div style="font-size:13px;opacity:.7;margin-bottom:8px">So với đơn giá tham chiếu {fmt_ppm2(base)} (mức trung bình của mô hình).
  Tác động được tính theo phần trăm và nhân dồn với nhau.</div>
  {bars if bars else '<i>Không có yếu tố nổi bật.</i>'}
</div>"""


def comps_table(loai, quan, phuong, du_an, dien_tich):
    c = COMPS[(COMPS["loai"] == loai) & (COMPS["district"] == quan)]
    if loai == LOAI_CC and du_an and du_an != "Không rõ" and (c["project"] == du_an).sum() >= 3:
        c = c[c["project"] == du_an]
    elif phuong and phuong != "Không rõ" and (c["ward"] == phuong).sum() >= 5:
        c = c[c["ward"] == phuong]
    c = c[(c["area"] >= dien_tich * 0.7) & (c["area"] <= dien_tich * 1.3)]
    if c.empty:
        return pd.DataFrame({"Thông báo": ["Chưa có tin rao tương tự trong dữ liệu."]})
    c = c.assign(_d=(np.log(c["area"] / dien_tich)).abs()).sort_values(["_d", "published"], ascending=[True, False]).head(6)
    return pd.DataFrame({
        "Tiêu đề tin rao": c["title"],
        "Phường": c["ward"],
        "Diện tích (m²)": c["area"].map(lambda v: vn(v, 0)),
        "Giá rao": c["price"].map(fmt_price),
        "Đơn giá": c["ppm2"].map(fmt_ppm2),
        "Ngày đăng": c["published"],
    })


# ----------------------------------------------------------------------------- giao diện
DISTRICTS = META["districts"]
WARDS = META["wards_by_district"]
PROJECTS = META["projects_by_district"]
KW_LABELS = [k[1] for k in KEYWORDS]


def on_loai(loai):
    is_cc = loai == LOAI_CC
    return (
        gr.update(visible=is_cc), gr.update(visible=not is_cc),
        gr.update(visible=not is_cc), gr.update(visible=not is_cc),
    )


def on_quan(quan):
    w = ["Không rõ"] + WARDS.get(quan, [])
    p = ["Không rõ"] + PROJECTS.get(quan, [])
    return gr.update(choices=w, value="Không rõ"), gr.update(choices=p, value="Không rõ")


def metrics_md():
    m = META["metrics"]
    lines = [
        "| Mô hình | Sai số trung vị | Sai số trung bình (MAPE) | Tỷ lệ dự đoán lệch ≤ 10% | Tỷ lệ lệch ≤ 20% |",
        "|---|---|---|---|---|",
    ]
    for r in m:
        lines.append(f"| {r['model']} | {vn(r['MdAPE'], 1)}% | {vn(r['MAPE'], 1)}% | {vn(r['within10'], 1)}% | {vn(r['within20'], 1)}% |")
    info = META["data_info"]
    return (
        f"Huấn luyện trên **{vn(info['n_train'], 0)}** tin rao bán nhà riêng và chung cư tại Hà Nội "
        f"({info['date_from']} – {info['date_to']}), sau khi làm sạch và loại tin trùng. "
        f"Kiểm tra trên **{vn(info['n_test'], 0)}** tin đăng **sau ngày {info['test_from']}**, là những tin mô hình chưa từng thấy, "
        f"giống tình huống định giá thực tế.\n\n" + "\n".join(lines) +
        "\n\n*Sai số tính trên giá rao bán, không phải giá giao dịch thực tế.*"
    )


d0 = DISTRICTS[0] if DISTRICTS else "Không rõ"
with gr.Blocks(title="Định giá nhà đất Hà Nội") as demo:
    gr.Markdown(
        "# 🏠 Định giá nhà đất Hà Nội bằng AI\n"
        "Nhập thông tin căn nhà hoặc căn hộ, mô hình sẽ ước tính giá dựa trên hàng trăm nghìn tin rao bán thực tế. "
        "Mô hình kết hợp **LightGBM** (đọc các thông số) và **PhoBERT** (đọc hiểu phần mô tả tiếng Việt)."
    )
    with gr.Row():
        with gr.Column(scale=5):
            loai = gr.Radio([LOAI_NHA, LOAI_CC], value=LOAI_NHA, label="Loại bất động sản")
            with gr.Row():
                quan = gr.Dropdown(DISTRICTS, value=d0, label="Quận / huyện", allow_custom_value=True)
                phuong = gr.Dropdown(["Không rõ"] + WARDS.get(d0, []), value="Không rõ", label="Phường / xã", allow_custom_value=True)
            du_an = gr.Dropdown(["Không rõ"] + PROJECTS.get(d0, []), value="Không rõ", label="Dự án chung cư", visible=False, allow_custom_value=True)
            with gr.Row():
                dien_tich = gr.Number(value=45, label="Diện tích (m²)", minimum=1)
                so_tang = gr.Number(value=4, label="Số tầng", minimum=1, precision=0)
                mat_tien = gr.Number(value=4, label="Mặt tiền (m)", minimum=0)
            with gr.Row():
                duong = gr.Number(value=3, label="Độ rộng ngõ / đường trước nhà (m)", minimum=0)
                ngu = gr.Number(value=3, label="Số phòng ngủ", minimum=0, precision=0)
                wc = gr.Number(value=3, label="Số WC", minimum=0, precision=0)
            with gr.Row():
                huong = gr.Dropdown(["Không rõ"] + HUONG, value="Không rõ", label="Hướng nhà")
                phap_ly = gr.Dropdown(PHAP_LY, value="Sổ đỏ/sổ hồng", label="Pháp lý")
            dac_diem = gr.CheckboxGroup(KW_LABELS, label="Đặc điểm nổi bật")
            mo_ta = gr.Textbox(
                lines=4, label="Mô tả thêm (không bắt buộc)",
                placeholder="Ví dụ: Nhà ngõ thông, cách phố 30m, gần trường học và chợ, hàng xóm dân trí cao...",
            )
            btn = gr.Button("Định giá", variant="primary", size="lg")
        with gr.Column(scale=6):
            out_card = gr.HTML()
            out_factors = gr.HTML()
            gr.Markdown("#### Tin rao tương tự trong khu vực")
            out_comps = gr.Dataframe(wrap=True, interactive=False)

    with gr.Accordion("Mô hình chính xác đến đâu?", open=False):
        gr.Markdown(metrics_md())
    gr.Markdown(
        "<small>⚠️ Kết quả chỉ mang tính tham khảo, dựa trên **giá rao bán** (thường cao hơn giá giao dịch thực tế). "
        "Dữ liệu: bộ [tinixai/vietnam-real-estates](https://huggingface.co/datasets/tinixai/vietnam-real-estates) (CC BY-NC 4.0). "
        "Lưu ý: dữ liệu còn dùng địa giới quận/huyện cũ (trước khi sáp nhập 01/07/2025).</small>"
    )

    inputs = [loai, quan, phuong, du_an, dien_tich, so_tang, mat_tien, duong, ngu, wc, huong, phap_ly, dac_diem, mo_ta]
    _q = lambda name: name if name in DISTRICTS else d0
    gr.Examples(
        label="Ví dụ mẫu (bấm để điền nhanh)",
        examples=[
            [LOAI_NHA, _q("Đống Đa"), "Không rõ", "Không rõ", 40, 5, 4, 3, 4, 4, "Không rõ", "Sổ đỏ/sổ hồng",
             ["Ô tô đỗ cửa / ngõ ô tô", "Kinh doanh được"], "Nhà ngõ thông, cách phố 20m, gần chợ và trường học."],
            [LOAI_CC, _q("Cầu Giấy"), "Không rõ", "Không rõ", 75, 1, 0, 0, 2, 2, "Đông Nam", "Sổ đỏ/sổ hồng",
             ["Full nội thất"], "Căn góc tầng trung, ban công Đông Nam, view thoáng, tiện ích đầy đủ."],
        ],
        inputs=inputs,
    )
    loai.change(on_loai, loai, [du_an, so_tang, mat_tien, duong])
    quan.change(on_quan, quan, [phuong, du_an])
    btn.click(predict, inputs, [out_card, out_factors, out_comps])

if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft(primary_hue="emerald"))


In [ ]:
%%writefile space/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch
transformers>=4.40
lightgbm
pandas
pyarrow
numpy


In [ ]:
%%writefile space/README.md
---
title: "Dinh gia nha dat Ha Noi"
emoji: 🏠
colorFrom: green
colorTo: blue
sdk: gradio
sdk_version: "6.32.0"
python_version: "3.11"
app_file: app.py
pinned: false
license: cc-by-nc-4.0
short_description: Định giá nhà & chung cư Hà Nội bằng LightGBM + PhoBERT
---

# 🏠 Định giá nhà đất Hà Nội bằng AI
Ước tính giá nhà riêng và chung cư Hà Nội từ thông số và mô tả tin rao (LightGBM + PhoBERT).


In [ ]:
%%writefile space/packages.txt
libgomp1


## 13. Đưa web demo lên Hugging Face Spaces 🚀

In [ ]:
if not LOCAL_TEST:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    repo_id = f"{HF_USER}/{SPACE_NAME}"
    api.create_repo(repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)
    api.upload_folder(folder_path="space", repo_id=repo_id, repo_type="space",
                      ignore_patterns=["__pycache__/*", "*.pyc"], commit_message="Cập nhật mô hình & giao diện")
    SPACE_URL = f"https://huggingface.co/spaces/{repo_id}"
    print("🎉 Web demo:", SPACE_URL)
    print("Lần đầu Space cần 3–5 phút để cài đặt. Vào link, chờ chữ 'Running' là dùng được.")
else:
    SPACE_URL = "https://huggingface.co/spaces/<ten-ban>/" + SPACE_NAME

## 14. Tạo gói file cho GitHub (README đã điền sẵn kết quả)

In [ ]:
def _row(r):
    return f"| {r['model']} | {r['MdAPE']:.1f}% | {r['MAPE']:.1f}% | {r['within10']:.1f}% | {r['within20']:.1f}% |".replace(".", ",")

_tbl = "\n".join(_row(r) for r in meta["metrics"])
_tbl_type = "\n".join(_row(r) for r in meta["metrics_by_type"])
_best = next(r for r in meta["metrics"] if r["model"] == FINAL_NAME)
_base = meta["metrics"][0]
_gain = (_base["MdAPE"] - _best["MdAPE"]) / _base["MdAPE"] * 100
_cov = ", ".join(f"{k}: {v:.0%}" for k, v in cov.items())
_info = meta["data_info"]
_n = lambda x: f"{x:,}".replace(",", ".")
_model_desc = ("**LightGBM** học từ thông số + từ khoá, kết hợp với **PhoBERT** (`vinai/phobert-base-v2`) được fine-tune để đọc hiểu mô tả tin rao tiếng Việt"
               if USE_PHOBERT else "**LightGBM** học từ thông số + từ khoá trích xuất từ mô tả")

README = f"""# 🏠 Định giá nhà đất Hà Nội bằng AI

**Demo trực tuyến:** {SPACE_URL}

Ứng dụng web giúp ước tính giá **nhà riêng** và **căn hộ chung cư** tại Hà Nội. Người dùng nhập vị trí, diện tích, số tầng, mặt tiền,
pháp lý, đặc điểm nổi bật và mô tả tự do. Mô hình trả về:

- **Giá ước tính** kèm **khoảng giá hợp lý 80%**
- **Đơn giá/m²** so với trung vị của phường, quận và toàn thành phố
- **Các yếu tố làm giá cao hoặc thấp hơn** (ví dụ: ô tô đỗ cửa +12%, chưa có sổ −15%)
- **Tin rao tương tự** trong khu vực để đối chiếu

Mô hình sử dụng {_model_desc}.

## Kết quả

Kiểm tra trên **{_n(_info['n_test'])} tin đăng sau ngày {_info['test_from']}**, là những tin mô hình chưa từng thấy khi huấn luyện
(chia theo thời gian, giống tình huống định giá thực tế).

| Mô hình | Sai số trung vị | MAPE | Lệch ≤ 10% | Lệch ≤ 20% |
|---|---|---|---|---|
{_tbl}

➡️ Mô hình cuối (**{FINAL_NAME}**) giảm **{_gain:.0f}%** sai số trung vị so với mô hình chỉ dùng thông số cơ bản.

Kết quả theo loại bất động sản (mô hình cuối):

| Loại | Sai số trung vị | MAPE | Lệch ≤ 10% | Lệch ≤ 20% |
|---|---|---|---|---|
{_tbl_type}

Độ phủ thực tế của khoảng giá 80% trên tập test: {_cov}.

![So sánh mô hình](outputs/so_sanh_mo_hinh.png)
![Dự đoán vs thực tế](outputs/du_doan_vs_thuc_te.png)
![Yếu tố quan trọng](outputs/muc_do_quan_trong.png)
![Đơn giá theo quận](outputs/don_gia_theo_quan.png)

## Dữ liệu

- Nguồn: [tinixai/vietnam-real-estates](https://huggingface.co/datasets/tinixai/vietnam-real-estates) (CC BY-NC 4.0), tin rao bán từ {_info['date_from']} đến {_info['date_to']}.
- Lọc Hà Nội, giữ nhà riêng và chung cư; loại tin thiếu giá hoặc diện tích, đơn giá bất thường, tin đăng trùng lặp, và ngoại lai theo từng quận.
- Sau khi làm sạch: **{_n(_info['n_train'] + _info['n_test'])}** tin.

## Phương pháp

1. **Làm sạch và loại trùng:** một tin thường được đăng lại nhiều lần; nếu không loại, tập test sẽ chứa tin đã thấy khi train và kết quả bị đẹp giả.
2. **Che giá trong văn bản:** nhiều tin ghi sẵn "giá 7,4 tỷ" trong tiêu đề; mọi con số giá được thay bằng `[giá]` trước khi đưa vào mô hình để tránh **rò rỉ dữ liệu**.
3. **Đặc trưng:**
   - Thông số: diện tích, số tầng, diện tích sàn, mặt tiền, độ rộng ngõ, số phòng, hướng nhà
   - Vị trí: quận, phường, dự án chung cư
   - Thời điểm đăng tin
   - 12 từ khoá nghiệp vụ trích từ mô tả: ô tô đỗ cửa, kinh doanh, lô góc, thang máy, pháp lý…
4. **PhoBERT:** fine-tune để dự đoán log(đơn giá) chỉ từ văn bản tiêu đề và mô tả.
5. **Xếp chồng mô hình (stacking) không rò rỉ:** PhoBERT học trên phần A của tập train; LightGBM học trên phần B, dùng dự đoán của PhoBERT làm một đặc trưng.
6. **Giải thích:** tác động của từng yếu tố được tính bằng SHAP (`pred_contrib` của LightGBM), quy đổi ra % giá.
7. **Khoảng giá 80%:** lấy theo phân phối sai số trên tập validation, tính riêng cho từng loại bất động sản.

## Cấu trúc thư mục

```
├── dinh_gia_nha_ha_noi.ipynb   # notebook Colab: dữ liệu → train → đánh giá → deploy
├── app.py                      # giao diện Gradio
├── features.py                 # xử lý dữ liệu & đặc trưng (dùng chung cho train và app)
├── requirements.txt
└── outputs/                    # biểu đồ kết quả
```

## Chạy lại

Mở notebook trên Google Colab, chọn GPU T4 rồi bấm **Runtime → Run all**. Notebook tự tải dữ liệu, huấn luyện và đưa web demo lên Hugging Face Spaces.

## Hạn chế & hướng phát triển

- Dữ liệu là **giá rao bán**, thường cao hơn giá giao dịch thực tế.
- Dữ liệu còn dùng địa giới quận/huyện cũ (trước khi sáp nhập đơn vị hành chính từ 01/07/2025).
- Chưa có toạ độ: có thể bổ sung khoảng cách tới trung tâm, ga metro, trường học bằng geocoding.
- Có thể tách từ tiếng Việt (VnCoreNLP) trước khi đưa vào PhoBERT, thử PhoBERT-large, hoặc thêm ảnh nhà (mô hình đa phương thức).
- Theo dõi biến động giá theo thời gian và tự động huấn luyện lại hằng tháng.
"""

os.makedirs("github_package/outputs", exist_ok=True)
with open("github_package/README.md", "w", encoding="utf-8") as f:
    f.write(README)
for fn in ["app.py", "features.py", "requirements.txt"]:
    shutil.copy(f"space/{fn}", f"github_package/{fn}")
for fn in os.listdir("outputs"):
    shutil.copy(f"outputs/{fn}", f"github_package/outputs/{fn}")
shutil.make_archive("github_package", "zip", "github_package")
print(README[:1500])
print("\n📦 Đã tạo github_package.zip — giải nén rồi tải toàn bộ lên GitHub (kèm file notebook này: File → Download → .ipynb).")


In [ ]:
if not LOCAL_TEST:
    from google.colab import files
    files.download("github_package.zip")

## 15. (Tuỳ chọn) Chạy thử giao diện ngay trong Colab

In [ ]:
!pip -q install "gradio==6.32.0"
try:
    import importlib, app
    importlib.reload(app)
    card, fac, comp = app.predict(LOAI_NHA, meta["districts"][0], "Không rõ", "Không rõ", 45, 4, 4, 3, 3, 3,
                                 "Không rõ", "Sổ đỏ/sổ hồng", ["Ô tô đỗ cửa / ngõ ô tô"], "nhà đẹp gần chợ")
    from IPython.display import HTML, display
    display(HTML(card)); display(HTML(fac)); display(comp)
except Exception as e:
    print("⚠️ Chạy thử app trong Colab lỗi (vẫn tiếp tục deploy):", repr(e))